# P01. Your first map of Princeton

*CEE420 Urban AI, Wednesday September 9, 2026*

Today in one line: make a map with your own hands, learn to prove a map is right, then delegate
a task to your coding agent and grade its work.

**Before you edit anything**: File > Save As, and save your copy as `01-first-map-mywork.ipynb`.
Copies named `-mywork` stay out of git, so next week's `git pull` will never fight your edits.

> 📚 The margin notes like this one map each cell to the self-study on-ramp. Doing the
> linked lesson this week turns today's magic words into things you can read and write yourself.

## Act 1. Layers

Every map you will make this semester starts the same way: read a file, look at it.

In [ ]:
import geopandas as gpd

boundary = gpd.read_file("data/princeton_boundary.geojson")
boundary.plot(figsize=(5, 5), color="#f5f0e6", edgecolor="black").set_axis_off()

> 📚 `read_file` and GeoDataFrames: [AutoGIS Lesson 2, GeoPandas: an introduction](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-2/overview.html).

In [ ]:
buildings = gpd.read_file("data/princeton_buildings.geojson")
food = gpd.read_file("data/princeton_food.geojson")

print(f"{len(buildings)} buildings, {len(food)} places to eat")
food.head(3)

> 📚 `len()`, variables, and tables: [Geo-Python Lesson 2](https://geo-python-site.readthedocs.io/en/latest/lessons/L2/overview.html) and [Lesson 5, pandas](https://geo-python-site.readthedocs.io/en/latest/lessons/L5/overview.html),
> the single most load-bearing lesson on the on-ramp.

In [ ]:
ax = boundary.plot(figsize=(8, 8), color="#f5f0e6", edgecolor="black", linewidth=1)
buildings.plot(ax=ax, color="#c8c2b8", linewidth=0)
food.plot(ax=ax, color="#e77500", markersize=12)
ax.set_title("Princeton: every building, every place to eat")
ax.set_axis_off()

Three places on campus live in a small table of coordinates. Turning a table of latitudes and
longitudes into something mappable is a move you will repeat all semester, and it is the first
thing you will need for your own town in A01.


In [ ]:
import pandas as pd

landmarks = pd.read_csv("data/landmarks.csv")
landmarks_gdf = gpd.GeoDataFrame(
    landmarks, geometry=gpd.points_from_xy(landmarks.lon, landmarks.lat), crs="EPSG:4326"
)
landmarks_gdf


## The wow cell

Static maps are for reports. This one you can touch: zoom in, click the dots,
find the cafes you actually go to, and find our classroom marker.

We only pass the boundary and the 92 food points here. Passing all 7,893 buildings to an
interactive map would write every one of their polygons into the page, about 3 MB of geometry, and
the result crawls. Rendering cost is a design constraint, remember it when you build your posters.

In [ ]:
webmap = boundary.explore(color="#e77500", style_kwds={"fill": False, "weight": 3})
webmap = food.explore(m=webmap, color="#e77500", tooltip=["name", "amenity"])
webmap = landmarks_gdf.explore(m=webmap, color="black", marker_kwds={"radius": 8}, tooltip="name")
webmap

## Change one thing

Pick a knob, turn it, re-run the cell. That loop is the whole craft.

- **Knob 1**: in the layered map above, change a color and the marker size.
- **Knob 2**: in the cell below, swap `cafe` for `pizza`... which does nothing. Look at the
  `amenity` values printed by `food.head()` and pick one that exists. Why did `pizza` fail?
- **Knob 3**: add your dorm to the landmarks map, coordinates from a long-press in your phone's map app.

In [ ]:
cafes = food[food["amenity"] == "cafe"]
print(f"{len(cafes)} of {len(food)} food places match")
cafes[["name", "amenity"]].head(10)

> ⭐ **Stretch cells** for those who finish early, no obligation. Later weeks explain both;
> today they are a taste of where this goes.

In [ ]:
# Stretch 1: New Jersey measures official areas in feet (EPSG:3424). Find the biggest building.
buildings_ft = buildings.to_crs("EPSG:3424")
biggest = buildings_ft.geometry.area.idxmax()
print(f"largest footprint: {buildings_ft.geometry.area.max():,.0f} square feet")
buildings.loc[[biggest]].plot(figsize=(4, 4)).set_axis_off()

In [ ]:
# Stretch 2: how many buildings stand within a 400 m walk of the Dinky station?
dinky = landmarks_gdf[landmarks_gdf.id == "dinky"].to_crs("EPSG:26918").geometry.iloc[0]
near_dinky = buildings.to_crs("EPSG:26918").intersects(dinky.buffer(400))
print(f"{near_dinky.sum()} buildings within 400 m of the Dinky")

## The mystery file

A colleague sends you `mystery_boundary.geojson`, labeled "Princeton boundary, ready to use".
It loads. It plots. It looks fine.

In [ ]:
mystery = gpd.read_file("data/mystery_boundary.geojson")
mystery.plot(figsize=(5, 5), color="#e8f0e6", edgecolor="black").set_axis_off()

**Is this our Princeton? Prove it with code, not with your eyes.** Three checks, type them yourself:

1. `mystery.crs`, what are the units?
2. `mystery.total_bounds`, where on Earth is this?
3. Its area in km2, reprojected to meters, against the reference below.

Reference numbers for the real Princeton: area **47.66 km2**, bounds
**[-74.7221, 40.3049, -74.6175, 40.3911]** (lon/lat). They are also printed on your handout and live in
`data/ground_truth.json`.

In [ ]:
# Type the three checks here.
# 1) mystery.crs
# 2) mystery.total_bounds
# 3) mystery.to_crs("EPSG:26918").geometry.area.iloc[0] / 1e6

**Doctrine of the day**: a map that renders is not a map that is right. Rendering is the
computer's job; being right is yours. These three lines are the price of admission, and they
work on any spatial dataset anyone ever hands you, including everything an AI hands you.

## The Duel, part 1: you, by hand

**How many food places are within a 400 m walk of our classroom (E225)?**

Straight-line distance today; real network walking distance is P07's business. You have five
minutes on the clock. The recipe in words: make E225 a point, put it in a CRS measured in
meters (EPSG:26918), buffer 400, count the food points inside. Most of the room will not
finish, and that is the point: feel the task in your fingers before you delegate it.

In [ ]:
# Your attempt. Ingredients: landmarks_gdf (E225 is id "e225"), food, .to_crs, .buffer, .within

## The Duel, part 2: your agent, audited

Now delegate exactly that task. Fill in the 3-line spec (also on your handout), give it to your
agent, and paste what comes back into the cell below.

```text
PLACE:  the point with id "e225" in data/landmarks.csv (EPSG:4326)
TASK:   count the food places in data/princeton_food.geojson within a 400 m
        straight-line buffer, computed in EPSG:26918 (meters, not degrees)
OUTPUT: the count as a number, and a map cell showing the circle on the food layer

Constraints: work only in this notebook, geopandas only, do not modify
earlier cells, no new packages, no internet access.
```

In [ ]:
# Paste your agent's code here and run it.

**The audit.** The agent's work is not done until you have checked it:

1. Is the buffer computed in a CRS measured in meters? (A 400 degree buffer swallows the Atlantic.)
2. Compare the count to the number on the board. If it differs, **can you say exactly why?** A
   different answer is not automatically a wrong one: measuring from a different point, counting
   bars as places to eat, or walking along streets instead of flying straight all change the number
   defensibly. An answer you cannot account for is the problem, not an answer that differs.
3. Is the circle actually on E225, not on some other point?

If your agent nailed it, say what each check would have caught. Graded work is trustworthy work.

## Your first AI note

Every submission in this course carries three lines. We write today's together; copy it here and
fill in your own numbers.

```text
AI note
Tools:    ...
They did: ...
Verified: ...
```

---
## Appendix: break glass in case of emergency

Reference solutions. Genuinely try first; these exist so nobody stays stuck.

<details>
<summary><b>The three checks, solved</b></summary>

```python
import geopandas as gpd
mystery = gpd.read_file("data/mystery_boundary.geojson")
print(mystery.crs)
print(mystery.total_bounds)
area_km2 = mystery.to_crs("EPSG:26918").geometry.area.iloc[0] / 1e6
print(f"{area_km2:.2f} km2")
# Real Princeton: ~47.66 km2, longitude near -74.7221, latitude near 40.3049.
# This file is somewhere else entirely.
```

</details>

<details>
<summary><b>The Duel, solved</b></summary>

```python
import geopandas as gpd
import pandas as pd
landmarks = pd.read_csv("data/landmarks.csv")
landmarks_gdf = gpd.GeoDataFrame(
    landmarks, geometry=gpd.points_from_xy(landmarks.lon, landmarks.lat), crs="EPSG:4326"
)
food = gpd.read_file("data/princeton_food.geojson")
e225 = landmarks_gdf[landmarks_gdf.id == "e225"].to_crs("EPSG:26918").geometry.iloc[0]
answer = int(food.to_crs("EPSG:26918").within(e225.buffer(400)).sum())
print(answer)
```

</details>